In [0]:
 from pyspark.sql.functions import udtf
 from datetime import datetime  , timedelta

@udtf(returnType = "date_value : string" ,useArrow = True)  #  (return field name = date_value  , datatype = string)
class DateExploder:      # udtf function
    def eval(self , start_date : str , expand_days : int ):  # override the exsting function
        current = datetime.strptime(start_date , "%Y-%m-%d")
        for i in range(expand_days):
            yield(current.strftime("%Y-%m-%d"))
            current += timedelta(days =1)




In [0]:
from pyspark.sql.functions import lit

DateExploder(lit("2025-07-27") ,lit(6))


In [0]:
DateExploder(lit("2025-07-27") ,lit(6)).display()

In [0]:
data_schema = "id int , name string ,join_date string"
data = [
        (101 , "Prasant"  , "2025-02-25"),
        (102 , "Sushant" , "2025-02-26")
]

df = spark.createDataFrame(data = data , schema = data_schema).alias("s")

In [0]:
df.lateralJoin(DateExploder("s.join_date" , lit(5)))\
    .select("id" , "name" , "join_date" , "date_value")\
        .display()    

In [0]:
spark.udtf.register("date_exploder" , DateExploder)

In [0]:
df.createOrReplaceTempView("student_view")

In [0]:
spark.sql("""
          select * from student_view ,  lateral date_exploder(join_date , 5)
          """).display()